# 12 — Operational Anomaly Alerts

Scans each branch's recent daily history for days where wait, volume, no-show, or completion broke
sharply (|z| ≥ 2) from that branch's own norm, and surfaces them as executive early-warning alerts.

**Outputs the `operational_anomalies` insight** consumed by the Executive dashboard.

## 1. Setup & daily branch metrics

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams.update({"axes.titleweight": "bold", "axes.titlesize": 12, "figure.dpi": 110})
pd.set_option("display.max_columns", 40)

# Lyne palette (matches the admin dashboards)
NAVY, STEEL, TEAL, RED, GOLD = "#2F5063", "#6E8AA6", "#2E7387", "#B23A4E", "#9A6B2E"
BLUES = sns.light_palette(NAVY, n_colors=6, reverse=True)

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE))
DOW = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
def hour_label(h): return f"{((int(h) + 11) % 12) + 1}{'am' if h < 12 else 'pm'}"
print("Ready.")

In [ ]:
from scripts import detect_operational_anomalies as an
from utils.metrics_utils import detect_anomalies

conn = an.connect()
daily = an.load_daily_branch(conn)
print(f"{len(daily):,} branch-days across {daily.branch_name.nunique()} branches")
daily[["branch_name", "date", "volume", "avg_wait", "no_show_rate", "completion_rate"]].tail()

## 2. Detected anomalies (recent window, ranked by severity)

In [ ]:
insights, _, _ = an.build_insights(daily)
biz = max(insights, key=lambda i: len(i["insight_data"]["anomalies"]))
rows = biz["insight_data"]["anomalies"]
print(biz["insight_data"]["summary"])
adf = pd.DataFrame(rows)
if len(adf):
    display(adf[["branch_name", "metric", "date", "value", "expected", "z_score", "severity"]])
else:
    print("No anomalies in the recent window.")

## 3. Where an anomaly showed up\nWait-time trend for the branch with the most alerts — anomalous days marked in red.

In [ ]:
if len(adf):
    branch = adf["branch_name"].value_counts().index[0]
    g = daily[daily["branch_name"] == branch].sort_values("date")
    an_dates = set(pd.to_datetime(adf[(adf.branch_name == branch) & (adf.metric == "wait time")]["date"]))
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(g["date"], g["avg_wait"], color=NAVY, lw=1.8, marker="o", ms=3)
    mask = g["date"].isin(an_dates)
    ax.scatter(g.loc[mask, "date"], g.loc[mask, "avg_wait"], color=RED, s=90, zorder=5, label="Anomaly")
    ax.axhline(g["avg_wait"].mean(), color=STEEL, ls="--", lw=1, label="Branch norm")
    ax.set_title(f"Average Wait — {branch} (anomalies flagged)"); ax.set_ylabel("Avg wait (min)")
    ax.legend(); plt.tight_layout(); plt.show()
else:
    print("No anomalies to plot.")

In [ ]:
if os.getenv("WRITE_DB") == "1":
    ins, gen, stale = an.build_insights(daily)
    an.upsert_insights(conn, ins, gen, stale, an.MODEL_VERSION)
    print(f"Upserted {len(ins)} operational_anomalies insight(s).")
else:
    print("Preview only — set WRITE_DB=1 to persist.")
conn.close()